A quick script to load the allegro model, evaluate and extract edge features across a dataset and fps sample to a smaller subset for training.

In [ ]:
from nequip.model.saved_models.load_utils import load_saved_model
import torch
from ase.io import read


In [ ]:
name = "input.extxyz"
save = False
atoms = read("test.extxyz", index="::")


In [ ]:
model = load_saved_model("argy_cf3.nequip.zip")
print(model)


In [ ]:
import torch
import numpy as np
from nequip.data import from_ase, AtomicDataDict
from nequip.data.transforms import ChemicalSpeciesToAtomTypeMapper, NeighborListTransform
from nequip.model.saved_models.load_utils import load_saved_model
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_saved_model("argy_cf3.nequip.zip")
model.to(device)
model.eval()

metadata = {}
for k, v in model.metadata.items():
    if isinstance(v, bytes):
        metadata[k] = v.decode("utf-8")
    else:
        metadata[k] = v

r_max = float(metadata["r_max"])
type_names = metadata["type_names"].split(" ")

type_mapper = ChemicalSpeciesToAtomTypeMapper(
    model_type_names=type_names,
    chemical_species_to_atom_type_map=None 
)
neighbor_list = NeighborListTransform(r_max=r_max)

captured_features = []
def hook_fn(module, inputs, output):
    captured_features.append(inputs[0].detach().cpu())

hook_handle = model.model.func.edge_readout.mlp_module.register_forward_hook(hook_fn)

descriptors = []
print(f"Processing {len(atoms)} structures...")

for atoms_i in tqdm(atoms):
    data = from_ase(atoms_i)
    data = type_mapper(data)
    data = neighbor_list(data)
    data = AtomicDataDict.to_(data, device)
    
    data["pos"].requires_grad = True
    
    if "batch" not in data:
        num_atoms = data["pos"].shape[0]
        data["batch"] = torch.zeros(num_atoms, dtype=torch.long, device=device)

    model(data)
    
    edge_feats = captured_features.pop() 
    structure_desc = torch.mean(edge_feats, dim=0).numpy()
    descriptors.append(structure_desc)

hook_handle.remove()
descriptors = np.array(descriptors)

print(f"Descriptors shape: {descriptors.shape}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

def farthest_point_sampling(X, n_samples):
    n_total = X.shape[0]
    selected = [np.random.randint(n_total)]
    min_dists = np.full(n_total, np.inf)

    for _ in range(n_samples - 1):
        current_dist = np.linalg.norm(X - X[selected[-1]], axis=1)
        min_dists = np.minimum(min_dists, current_dist)
        selected.append(np.argmax(min_dists))

    return np.array(selected)

if len(descriptors) < 2:
    fps_indices = np.arange(len(descriptors))
    X_pca_all = np.zeros((len(descriptors), 2))
    X_tsne = np.zeros((len(descriptors), 2))
else:
    scaler = StandardScaler()
    descriptors_scaled = scaler.fit_transform(descriptors)

    n_comps = min(50, len(descriptors), descriptors.shape[1])
    pca = PCA(n_components=n_comps)
    X_pca_all = pca.fit_transform(descriptors_scaled)

    perp = min(30, len(descriptors) - 1)
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, init='pca', learning_rate='auto')
    X_tsne = tsne.fit_transform(X_pca_all)

    n_select = min(100, len(descriptors))
    fps_indices = farthest_point_sampling(X_tsne, n_select)

def ensure_2d(X_data):
    if X_data.shape[1] < 2:
        X_plot = np.zeros((X_data.shape[0], 2))
        X_plot[:, 0] = X_data[:, 0]
        return X_plot
    return X_data

def style_axis(ax):
    ax.tick_params(axis='both', which='major', labelsize=11, length=4)
    ax.grid(True, linestyle='--', linewidth=0.7, alpha=0.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def make_embedding_plot(X_data, selected_idx, title, xlabel, ylabel):
    X_plot = ensure_2d(X_data)
    fig, ax = plt.subplots(figsize=(7.2, 6.0), dpi=180)

    ax.scatter(
        X_plot[:, 0], X_plot[:, 1],
        c='0.45', alpha=0.42, s=26, linewidths=0, label='All structures'
    )
    ax.scatter(
        X_plot[selected_idx, 0], X_plot[selected_idx, 1],
        c='#c62828', edgecolors='white', linewidth=0.8, s=92,
        label='FPS selected', zorder=3
    )

    ax.set_title(title, fontsize=14, pad=10)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    style_axis(ax)
    ax.legend(frameon=False, fontsize=10, loc='best')
    fig.tight_layout()
    plt.show()

make_embedding_plot(X_pca_all[:, :2], fps_indices, "PCA Projection", "PC 1", "PC 2")
make_embedding_plot(X_tsne, fps_indices, "t-SNE Embedding", "Dim 1", "Dim 2")

In [ ]:
from ase.io import write

if save:
    selected_atoms = [atoms[i] for i in fps_indices]
    write(f"{name}.extxyz", selected_atoms)